# 🧥 Glimpse — Fashion Domain

This notebook walks through the full pipeline for the **fashion** domain:
1. Load images from the Fashion Product Images dataset
2. Embed them with CLIP
3. Build a FAISS index for visual search
4. Cluster with UMAP + HDBSCAN to find aesthetic groups
5. Score trend momentum across clusters

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

from glimpse.embedder import Embedder
from glimpse.index import ImageIndex
from glimpse.cluster import AestheticClusterer
from glimpse.trends import TrendAnalyser

## 1. Load Dataset

Download the Fashion Product Images (Small) dataset from Kaggle first:
```bash
bash data/download.sh fashion
```

In [ ]:
DATA_DIR = Path('../data/raw/fashion')
IMAGE_DIR = DATA_DIR / 'images'

# Load metadata
styles_df = pd.read_csv(DATA_DIR / 'styles.csv', on_error='warn')
print(f'Loaded {len(styles_df)} items')
styles_df.head()

In [ ]:
# Get valid image paths
image_paths = []
metadata = []

for _, row in styles_df.iterrows():
    img_path = IMAGE_DIR / f"{row['id']}.jpg"
    if img_path.exists():
        image_paths.append(str(img_path))
        metadata.append({
            'path': str(img_path),
            'category': row.get('masterCategory', 'unknown'),
            'subcategory': row.get('subCategory', 'unknown'),
            'article': row.get('articleType', 'unknown'),
            'colour': row.get('baseColour', 'unknown'),
            'season': row.get('season', 'unknown'),
            'year': row.get('year', 0),
        })

print(f'Found {len(image_paths)} valid images')

## 2. Embed with CLIP

In [ ]:
embedder = Embedder(model_name='ViT-B/32')

# Embed all images (this may take a while on CPU)
# Tip: use a GPU or reduce the dataset for prototyping
embeddings = embedder.embed_batch(image_paths[:2000], batch_size=64)
print(f'Embeddings shape: {embeddings.shape}')

## 3. Build FAISS Index & Search

In [ ]:
index = ImageIndex(dim=512)
index.add(embeddings, metadata[:2000])
print(f'Index size: {index.size}')

# Search with the first image as a query
query_embedding = embeddings[0]
results = index.search(query_embedding, k=5)

print(f'\nQuery: {metadata[0]["article"]} ({metadata[0]["colour"]})')
print('Top matches:')
for r in results:
    print(f'  {r["article"]:20s} {r["colour"]:15s} score={r["score"]:.3f}')

## 4. Cluster Aesthetics

In [ ]:
clusterer = AestheticClusterer(min_cluster_size=20)
summary = clusterer.fit(embeddings)
print(f'Found {summary["n_clusters"]} clusters ({summary["n_noise_points"]} noise points)')

# Plot the 2D map
coords = clusterer.get_coordinates()
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    coords[:, 0], coords[:, 1],
    c=clusterer.labels, cmap='Spectral',
    s=5, alpha=0.6
)
plt.colorbar(scatter, label='Cluster')
plt.title('Fashion Aesthetic Map (CLIP + UMAP + HDBSCAN)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.show()

## 5. Trend Analysis

In [ ]:
# Use 'year' as a proxy for temporal trends
years = np.array([m.get('year', 0) for m in metadata[:2000]])

scores = TrendAnalyser.score_by_recency(
    labels=clusterer.labels,
    timestamps=years,
    n_recent=200
)

print('Cluster trend scores (> 1.0 = trending up):')
for cluster_id, score in scores.items():
    direction = '📈' if score > 1.2 else '📉' if score < 0.8 else '➡️'
    print(f'  Cluster {cluster_id:3d}: {score:.2f} {direction}')